In [6]:
import yaml
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

In [13]:
toe_to_MWh = 11.630

In [9]:
with open(Path.cwd().parent / 'config.basicrun.yaml', 'r') as f:
    params = yaml.safe_load(f)['industry']

In [21]:
sheet_names = {
    "Iron and steel": "ISI",
    "Chemicals Industry": "CHI",
    "Non-metallic mineral products": "NMM",
    "Pulp, paper and printing": "PPA",
    "Food, beverages and tobacco": "FBT",
    "Non Ferrous Metals": "NFM",
    "Transport equipment": "TRE",
    "Machinery equipment": "MAE",
    "Textiles and leather": "TEL",
    "Wood and wood products": "WWP",
    "Other industrial sectors": "OIS",
}
index = [
    "<100",
    "100-200",
    "200-500",
    ">500",
]
notes = {}

In [17]:
path = Path.cwd().parent / 'data' / 'jrc-idees-2021'

In [18]:
def load_idees_data(sector, country="EU27"):
    suffixes = {"out": "", "fec": "_fec", "ued": "_ued", "emi": "_emi"}
    sheets = {k: sheet_names[sector] + v for k, v in suffixes.items()}

    def usecols(x):
        return isinstance(x, str) or x == 2021

    idees = pd.read_excel(
        f"{path}/{country}/JRC-IDEES-2021_Industry_{country}.xlsx",
        sheet_name=list(sheets.values()),
        index_col=0,
        header=0,
        usecols=usecols,
    )

    for k, v in sheets.items():
        idees[k] = idees.pop(v).squeeze()
        idees[k] = idees[k][2021]

    return idees

In [ ]:
def iron_and_steel():
    sector = "Iron and steel"
    idees = load_idees_data(sector)

    df = pd.DataFrame(index=index)

    ## Electric arc

    sector = "Electric arc"
    df[sector] = 0.0

    s_fec = idees["fec"][52:68]
    assert s_fec.index[0] == sector

    df.at["<100", sector] += s_fec.loc["Low-enthalpy heat"]

    subsector = "Steel: Smelters"
    s_fec = idees["fec"][63:68]
    s_ued = idees["ued"][63:68]
    assert s_fec.index[0] == subsector
    assert s_ued.index[0] == subsector

    # efficiency changes due to transforming all the smelters into methane
    key = "Natural gas and biogas"
    eff_met = s_ued.loc[key] / s_fec.loc[key]

    df.at[">500", sector] += s_ued[subsector] / eff_met

    # conversion to MWh/t material
    s_out = idees["out"][7:8]

    df.loc[:, sector] = df.loc[:, sector] * toe_to_MWh / s_out[sector]

    print('after electric arc')
    print(df)

    ## DRI + Electric arc
    # For primary route: DRI with H2 + EAF

    sector = "DRI + Electric arc"

    df[sector] = df["Electric arc"]

    # add H2 consumption for DRI at 1.7 MWh H2 /ton steel
    df.at[">500", sector] = params["H2_DRI"]

    print('after DRI + Electric arc')
    print(df)

    ## Integrated steelworks
    # could be used in combination with CCS)
    # Assume existing fuels are kept, except for furnaces, refining, rolling, finishing
    # Ignore 'derived gases' since these are top gases from furnaces

    sector = "Integrated steelworks"

    df[sector] = 0.0

    s_fec = idees["fec"][3:50]
    assert s_fec.index[0] == sector

    df.loc["<100", sector] += s_fec["Low-enthalpy heat"]

    subsector = "Steel: Sinter/Pellet-making"

    s_fec = idees["fec"][14:20]
    s_ued = idees["ued"][14:20]
    assert s_fec.index[0] == subsector
    assert s_ued.index[0] == subsector

    sel = ["Natural gas and biogas", "Fuel oil", "Solids"]
    df.loc[">500", sector] += s_fec[sel].sum()

    subsector = "Steel: Blast /Basic oxygen furnace"

    s_fec = idees["fec"][20:26]
    s_ued = idees["ued"][20:26]
    assert s_fec.index[0] == subsector
    assert s_ued.index[0] == subsector

    sel = ["Natural gas and biogas", "Fuel oil", "Solids", "Coke"]
    df.loc[">500", sector] += s_fec[sel].sum()

    notes[sector + " >500"] = "The coke is used for heat, but is not fully replacabale in Integrated Steelworks"

    s_out = idees["out"][6:7]
    assert s_out.index[0] == sector

    # final energy consumption MWh/t material
    df.loc[:, sector] = df.loc[:, sector] * toe_to_MWh / s_out[sector]

    print('after Integrated steelworks')
    print(df)


In [29]:
iron_and_steel()

after electric arc
         Electric arc
<100         0.004876
100-200      0.000000
200-500      0.000000
>500         0.161758
after DRI + Electric arc
         Electric arc  DRI + Electric arc
<100         0.004876            0.004876
100-200      0.000000            0.000000
200-500      0.000000            0.000000
>500         0.161758            1.700000
after Integrated steelworks
         Electric arc  DRI + Electric arc  Integrated steelworks
<100         0.004876            0.004876               0.005924
100-200      0.000000            0.000000               0.000000
200-500      0.000000            0.000000               0.000000
>500         0.161758            1.700000               0.655360
